# Artist/Album Supervision for VarLen Semantic IDs

Короткий notebook для запуска course-project эксперимента.

**Research question:** может ли supervision по артисту и альбому улучшить variable-length semantic IDs для музыкальной рекомендации?

Сравниваем три варианта на уменьшенном Yambda subset:

1. `original` — обычный VarLen dVAE baseline.
2. `aux` — VarLen dVAE + auxiliary artist/album classification loss по полному SID-представлению.
3. `prefix` — VarLen dVAE + prefix-level supervision: первый токен тянем к artist cluster, первые два токена к album cluster.

Notebook не содержит отдельной логики эксперимента: он вызывает скрипты из `scripts/project/`, чтобы запуск оставался воспроизводимым из CLI.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

ROOT = Path.cwd()
if not (ROOT / "README.md").exists():
    raise RuntimeError("Run this notebook from the repository root")

def run(cmd, *, check=True):
    print("+", " ".join(cmd))
    return subprocess.run(cmd, cwd=ROOT, check=check)

PY = sys.executable
DATA_DIR = ROOT / "data" / "project" / "yambda"
RESULTS_DIR = ROOT / "results" / "project"
DATA_DIR, RESULTS_DIR

## 1. Download Yambda inputs

Эту ячейку нужно запускать только если `data/yambda/` еще не содержит:

- `interactions.parquet`
- `embeddings.parquet`
- `artist_item_mapping.parquet`
- `album_item_mapping.parquet`

Скачивание требует `datasets` и доступ к Hugging Face.

In [ ]:
# Optional: uncomment for a fresh machine.
# from scripts.data.yambda import download, download_metadata
# download(dst_dir="./data/yambda")
# download_metadata(dst_dir="./data/yambda")

## 2. Build project subset and metadata

По умолчанию берется до 5000 самых активных пользователей, до 3M interactions и до 20k core items. Для совсем маленькой проверки можно уменьшить параметры прямо в команде.

In [ ]:
run([
    PY, "-m", "scripts.project.build_yambda_subset",
    "--src-dir", "./data/yambda",
    "--dst-dir", "./data/project/yambda",
    "--num-users", "5000",
    "--max-interactions", "3000000",
    "--max-core-items", "20000",
])

run([
    PY, "-m", "scripts.project.build_artist_album_metadata",
    "--src-dir", "./data/yambda",
    "--data-dir", "./data/project/yambda",
    "--num-artist-classes", "512",
    "--num-album-classes", "1024",
])

## 3. Tiny smoke run

Если окружение и GPU готовы, сначала прогоняем дешевую baseline-проверку. В текущем lightweight окружении автора smoke test может не запускаться из-за отсутствующих `torch/polars/pyarrow/PyYAML`.

In [ ]:
run([PY, "-m", "scripts.train_dvae", "--config", "configs/project/original_varlen_dvae_tiny.yaml"])
run([PY, "-m", "scripts.train_seqrec", "--config", "configs/project/seqrec_original_tiny.yaml"])
run([
    PY, "-m", "scripts.project.analyze_prefix_purity",
    "--sids", "./results/project/original_tiny/sids.parquet",
    "--output", "./results/project/original_tiny/prefix_purity.json",
])

## 4. Full experiment runs

Каждый метод пишет:

- `results/project/<method>/sids.parquet`
- `results/project/<method>/dvae_metrics.json`
- `results/project/<method>/seqrec_summary.json`
- `results/project/<method>/prefix_purity.json`

Если времени мало, можно запустить `prefix` только до structural diagnostics: `--stages dvae,purity`.

In [ ]:
for method in ["original", "aux", "prefix"]:
    run([PY, "-m", "scripts.project.run_experiment", "--method", method])

## 5. Collect comparison table

Собираем компактную таблицу с seqrec metrics, purity и collision statistics.

In [ ]:
run([PY, "-m", "scripts.project.collect_results"])

comparison_path = RESULTS_DIR / "comparison.json"
if comparison_path.exists():
    comparison = json.loads(comparison_path.read_text())
    comparison
else:
    print("comparison.json is not created yet")

## Useful partial reruns

```bash
python3 -m scripts.project.run_experiment --method original --stages dvae,purity
python3 -m scripts.project.run_experiment --method aux --stages seqrec
python3 -m scripts.project.run_experiment --method prefix --stages dvae,purity
python3 -m scripts.project.collect_results
```

Fallback policy: если полный seqrec слишком долгий, оставляем end-to-end для `original` и `aux`, а для `prefix` репортим structural metrics: artist/album prefix purity, collisions, mean SID length.